In [1]:
import os
os.chdir('..')
os.chdir('..')

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import json

from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

d:\anaconda3\envs\pytorchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from src.data.dataset import LODODataset
from src.data.transform import train_transform, eval_transform
from src.utils.utils import get_regions, prepare_lodo_classes
from src.model_factory import create_model

In [4]:
class Trainer:
    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        test_loader,
        criterion,
        optimizer,
        scheduler=None,
        use_amp=True,
        output_dir="./outputs",
        model_key="model",
        dataset_name="dataset",
        class_to_idx=None,
        idx_to_class=None,
    ):
        self.model = model

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.use_amp = use_amp and self.device.type == "cuda"
        self.scaler = GradScaler(enabled=self.use_amp)

        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.model_key = model_key
        self.dataset_name = dataset_name

        self.class_to_idx = class_to_idx
        self.idx_to_class = idx_to_class

        self.best_val_acc = 0.0
        self.best_epoch = 0
        self.history = []

    def get_model_state_dict(self):
        if isinstance(self.model, nn.DataParallel):
            return self.model.module.state_dict()
        return self.model.state_dict()

    def load_model_state_dict(self, state_dict):
        if isinstance(self.model, nn.DataParallel):
            self.model.module.load_state_dict(state_dict)
        else:
            self.model.load_state_dict(state_dict)

    def _train_epoch(self, epoch):
        self.model.train()

        total_loss = 0.0
        total_correct = 0
        total_images = 0

        pbar = tqdm(
            self.train_loader,
            desc=f"Train epoch {epoch}",
            dynamic_ncols=True,
        )

        for images, labels, regions in pbar:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)

            # Object classification only
            with autocast(enabled=self.use_amp):
                logits = self.model(images)
                loss = self.criterion(logits, labels)

            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            preds = logits.argmax(dim=1)
            correct = (preds == labels).sum().item()

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += correct
            total_images += batch_size

            avg_loss = total_loss / total_images
            avg_acc = total_correct / total_images

            pbar.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "object_acc": f"{avg_acc:.4f}",
            })

        return {
            "loss": total_loss / total_images,
            "object_acc": total_correct / total_images,
        }

    @torch.no_grad()
    def evaluate(self, loader, split_name="val"):
        self.model.eval()

        total_loss = 0.0
        total_correct = 0
        total_images = 0

        region_correct = defaultdict(int)
        region_total = defaultdict(int)

        pbar = tqdm(
            loader,
            desc=f"Evaluate {split_name}",
            dynamic_ncols=True,
        )

        for images, labels, regions in pbar:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            # Object classification only
            with autocast(enabled=self.use_amp):
                logits = self.model(images)
                loss = self.criterion(logits, labels)

            preds = logits.argmax(dim=1)
            correct_mask = preds == labels

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += correct_mask.sum().item()
            total_images += batch_size

            for region, is_correct in zip(regions, correct_mask.cpu().tolist()):
                region_total[region] += 1
                region_correct[region] += int(is_correct)

            avg_acc = total_correct / total_images
            pbar.set_postfix({
                "object_acc": f"{avg_acc:.4f}",
            })

        avg_loss = total_loss / total_images
        avg_acc = total_correct / total_images

        region_object_acc = {}
        for region in sorted(region_total.keys()):
            region_object_acc[region] = region_correct[region] / region_total[region]

        if len(region_object_acc) > 0:
            best_region = max(region_object_acc, key=region_object_acc.get)
            worst_region = min(region_object_acc, key=region_object_acc.get)

            best_region_acc = region_object_acc[best_region]
            worst_region_acc = region_object_acc[worst_region]
            region_bias_gap = best_region_acc - worst_region_acc
        else:
            best_region = None
            worst_region = None
            best_region_acc = None
            worst_region_acc = None
            region_bias_gap = None

        return {
            "loss": avg_loss,
            "object_acc": avg_acc,
            "region_object_acc": region_object_acc,
            "best_region": best_region,
            "worst_region": worst_region,
            "best_region_acc": best_region_acc,
            "worst_region_acc": worst_region_acc,
            "region_bias_gap": region_bias_gap,
        }

    def save_checkpoint(self, epoch, val_results):
        save_path = self.output_dir / "best_model.pth"

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.get_model_state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "best_val_acc": self.best_val_acc,
            "class_to_idx": self.class_to_idx,
            "idx_to_class": self.idx_to_class,
            "model_key": self.model_key,
            "dataset_name": self.dataset_name,
            "val_results": val_results,
        }

        if self.scheduler is not None:
            checkpoint["scheduler_state_dict"] = self.scheduler.state_dict()

        torch.save(checkpoint, save_path)
        print("Saved best model:", save_path)

    def train(self, epochs):
        for epoch in range(1, epochs + 1):
            print("=" * 80)
            print(f"Epoch {epoch}/{epochs}")
            print("=" * 80)

            train_results = self._train_epoch(epoch)
            val_results = self.evaluate(self.val_loader, split_name="val")

            if self.scheduler is not None:
                self.scheduler.step()

            print(f"Train Loss:        {train_results['loss']:.4f}")
            print(f"Train Object Acc:  {train_results['object_acc']:.4f}")
            print(f"Val Loss:          {val_results['loss']:.4f}")
            print(f"Val Object Acc:    {val_results['object_acc']:.4f}")
            print(f"Region Bias Gap:   {val_results['region_bias_gap']}")

            print("\nObject accuracy per region:")
            for region, acc in val_results["region_object_acc"].items():
                print(f"  {region}: {acc:.4f}")

            epoch_log = {
                "epoch": epoch,
                "lr": self.optimizer.param_groups[0]["lr"],
                "train_loss": train_results["loss"],
                "train_object_acc": train_results["object_acc"],
                "val_loss": val_results["loss"],
                "val_object_acc": val_results["object_acc"],
                "val_region_object_acc": val_results["region_object_acc"],
                "val_best_region": val_results["best_region"],
                "val_worst_region": val_results["worst_region"],
                "val_region_bias_gap": val_results["region_bias_gap"],
            }

            self.history.append(epoch_log)

            if val_results["object_acc"] > self.best_val_acc:
                self.best_val_acc = val_results["object_acc"]
                self.best_epoch = epoch
                self.save_checkpoint(epoch, val_results)

            self.save_history()

        print()
        print("Training finished")
        print(f"Best epoch: {self.best_epoch}")
        print(f"Best val object acc: {self.best_val_acc:.4f}")

    def test(self):
        ckpt_path = self.output_dir / "best_model.pth"
        checkpoint = torch.load(ckpt_path, map_location=self.device)

        self.load_model_state_dict(checkpoint["model_state_dict"])

        test_results = self.evaluate(
            self.test_loader,
            split_name="test",
        )

        print()
        print("FINAL TEST RESULTS")
        print()
        print(f"Test Object Acc:   {test_results['object_acc']:.4f}")
        print(f"Best Region:       {test_results['best_region']} ({test_results['best_region_acc']})")
        print(f"Worst Region:      {test_results['worst_region']} ({test_results['worst_region_acc']})")
        print(f"Region Bias Gap:   {test_results['region_bias_gap']}")

        print("\nObject accuracy per region:")
        for region, acc in test_results["region_object_acc"].items():
            print(f"  {region}: {acc:.4f}")

        self.save_json(test_results, "test_results.json")

        return test_results

    def save_history(self):
        self.save_json(self.history, "history.json")

    def save_json(self, obj, filename):
        path = self.output_dir / filename

        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

In [5]:
def create_lodo_dataloaders(
    target_region,
    all_regions,
    train_root,
    val_root,
    test_root,
    class_to_idx,
    allowed_classes,
    batch_size,
    num_workers,
):
    source_regions = [
        region for region in all_regions
        if region != target_region
    ]

    train_dataset = LODODataset(
        root_path=train_root,
        class_to_idx=class_to_idx,
        transform=train_transform(),
        include_regions=source_regions,
        allowed_classes=allowed_classes,
    )

    val_dataset = LODODataset(
        root_path=val_root,
        class_to_idx=class_to_idx,
        transform=eval_transform(),
        include_regions=source_regions,
        allowed_classes=allowed_classes,
    )

    test_dataset = LODODataset(
        root_path=test_root,
        class_to_idx=class_to_idx,
        transform=eval_transform(),
        include_regions=[target_region],
        allowed_classes=allowed_classes,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    print("Target region:", target_region)
    print("Source regions:", source_regions)
    print("Train images:", len(train_dataset))
    print("Val images:", len(val_dataset))
    print("Test images:", len(test_dataset))

    return train_loader, val_loader, test_loader, source_regions

In [6]:
train_path = r"D:\Project\DAP_paper\datasets\processed\GeoDE_split\train"
test_path = r"D:\Project\DAP_paper\datasets\processed\GeoDE_split\test"
val_path = r"D:\Project\DAP_paper\datasets\processed\GeoDE_split\val"

In [7]:
all_regions = get_regions(train_path)

use_common_classes_for_lodo = False

model = create_model(model_name="convnextv2_base.fcmae_ft_in22k_in1k",
                     num_classes=40,
                     device="cuda")

class_to_idx, idx_to_class, allowed_classes = prepare_lodo_classes(
    train_root=train_path,
    use_common_classes=use_common_classes_for_lodo
)

target_region = "Africa"
batch_size = 32
num_workers = 2
lr = 1e-4
T_max = 50,
eta_min = 1e-6

source_regions = [region for region in all_regions if region != target_region]

train_dataset = LODODataset(
    root_path=train_path,
    class_to_idx=class_to_idx,
    transform=train_transform(),
    include_regions=source_regions,
    allowed_classes=allowed_classes
)

train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
)

val_dataset = LODODataset(
    root_path=val_path,
    class_to_idx=class_to_idx,
    transform=eval_transform(),
    include_regions=source_regions,
    allowed_classes=allowed_classes
)

val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
)

test_dataset = LODODataset(
        root_path=test_path,
        class_to_idx=class_to_idx,
        transform=eval_transform(),
        include_regions=[target_region],
        allowed_classes=allowed_classes,
)

test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = Adam(
    model.parameters(),
    lr=lr
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=T_max,
    eta_min=eta_min
)

All classes: 40
Common classes: 40
Final classes: 40


In [8]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    class_to_idx=class_to_idx,
    idx_to_class=idx_to_class,
)

C:\Users\VUONG\AppData\Local\Temp\ipykernel_21392\840720462.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.use_amp)
